In [ ]:
#this didnt reveale something very new, but confiremd a hypophsis he had on choosing the best plan for the curreculum pregressive
# learning, we take run through different plans and evaluate the model on each with validation set, and eventually test on test set.
#feautre idea, employ RL! have an agent with some reward function travle the parameter space(maybe add learning rate and epoch number)
#and search for the best plan, might revele something unexpected as the plan we check here are from a specific function family
import pandas as pd
import numpy as np
from tqdm import tqdm_pandas
from tqdm.notebook import tqdm
from transformers import BertModel, BertTokenizerFast, Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer, BertForSequenceClassification, BertTokenizer
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,  roc_auc_score, fbeta_score
from scipy.stats import norm
import warnings
import random

warnings.filterwarnings("ignore")

/home/ronfr/.conda/envs/alephbert_eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# hyperparameters
batch_size = 16
learning_rate = 2e-5

In [3]:
model_name = 'onlplab/alephbert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name , num_labels=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


***loading conv info***

In [4]:
conv_info_path = 'trainDatasets/conv_info.csv'
conv_info_df = pd.read_csv(conv_info_path)
conv_info_df['engagement_id'] = conv_info_df['engagement_id'].astype(str)
conv_info_df = conv_info_df.rename(columns={'gsr': 'label'})
train_conv_df, test_conv_df = train_test_split(conv_info_df, test_size=0.2, stratify=conv_info_df['label'],random_state=42)
train_conv_df, val_conv_df = train_test_split(train_conv_df, test_size=0.25, stratify=train_conv_df['label'],random_state=42)

***loading llama labled messages***

In [5]:
messages_with_lables_path = 'trainDatasets/combined_riskfree_riskfull_messages_syntatic_fixed.csv'

messages_with_lables_df = pd.read_csv(messages_with_lables_path)

messages_with_lables_df['engagement_id'] = messages_with_lables_df['engagement_id'].astype(str)
messages_with_lables_df = messages_with_lables_df[messages_with_lables_df['anonymized'].notna()]
messages_with_lables_df['name'] = messages_with_lables_df['name'].fillna('-')

#split to train and test base on conversation split
train_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(train_conv_df["engagement_id"])]
val_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(val_conv_df["engagement_id"])]
test_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(test_conv_df["engagement_id"])]

***loading all messages***

In [6]:
all_messages_path = 'trainDatasets/messages_anonymized.csv'

all_messages_df = pd.read_csv(all_messages_path)

all_messages_df['engagement_id'] = all_messages_df['engagement_id'].astype(str)
all_messages_df = all_messages_df[all_messages_df['anonymized'].notna()]
all_messages_df['name'] = all_messages_df['name'].fillna('-')
all_messages_df = all_messages_df[all_messages_df["seeker"]]

#split to train and test base on conversation split and concat messages
train_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(train_conv_df["engagement_id"])]
train_all_messages_df = train_all_messages_df.merge(train_conv_df , on="engagement_id")
train_all_messages_df = train_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()

val_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(val_conv_df["engagement_id"])]
val_all_messages_df = val_all_messages_df.merge(val_conv_df , on="engagement_id")
val_all_messages_df = val_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()

test_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(test_conv_df["engagement_id"])]
test_all_messages_df = test_all_messages_df.merge(test_conv_df , on="engagement_id")
test_all_messages_df = test_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()


## creating Dataset objects for train and test

In [7]:
# mapping the text into inputs that fits the model
def tokenize(batch):
    return tokenizer(batch['anonymized'], padding='max_length', truncation=True, max_length=512)

def make_weighted_train_loaders(dfs, weights, tokenize):
    #make sampled df
    acc_train = pd.DataFrame(data= {})
    for df, weight in zip(dfs,weights):
        sample = df.sample(frac = weight ,random_state=42)
        acc_train = pd.concat([acc_train,sample[["anonymized" , "label"]]])
    
    #create datasets
    train_dataset = Dataset.from_pandas(acc_train)
    train_dataset = train_dataset.map(tokenize, batched=True, batch_size=16)
    train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    return train_loader



def make_test_loader(df, tokenize):
    dataset = Dataset.from_pandas(df)
    dataset = dataset.map(tokenize, batched=True, batch_size=16)
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return test_loader

In [8]:
#labled_messages_train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , [1,0], tokenize)
#all_messages_train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , [0,1], tokenize)

labled_messages_test_loader = make_test_loader(test_labled_messages_df, tokenize)
all_messages_test_loader = make_test_loader(test_all_messages_df, tokenize)

labled_messages_val_loader = make_test_loader(val_labled_messages_df, tokenize)
all_messages_val_loader = make_test_loader(val_all_messages_df, tokenize)

Map:  91%|█████████▏| 76576/83832 [00:23<00:02, 3250.86 examples/s]


KeyboardInterrupt: 

## train model

***train on labled messages***

In [9]:
def general_trainer(train_loader,optimizer, loss_fn=None):
    bert_model.train()
    
    total_batches = len(train_loader)
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = bert_model(input_ids, attention_mask=attention_mask, labels=labels)
        if loss_fn is None:
            loss = outputs.loss
        else:
            loss = loss_fn(outputs, labels)
        
        loss.backward()
        optimizer.step()
        #progress_bar.update(1)
        if (i + 1) % 1000 == 0:
            batches_done = i + 1
            batches_left = total_batches - batches_done
            print(f"Batch {batches_done}/{total_batches} completed. {batches_left} remaining.")

In [10]:
def sigmoid_emphasis_pairs(n=6, k=10):
    t = np.linspace(0, 1, n)
    y = 1 / (1 + np.exp(-k * (t - 0.5)))
    x = 1 - y
    return list(zip(x, y))

def eval(test_loader):
    bert_model.eval()
    labels = []
    preds = []
    pred_probs = []
    
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label = batch['label'].to(device)
    
        with torch.no_grad():
            outputs = bert_model(input_ids, attention_mask=attention_mask)
    
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)
        predictions = torch.argmax(logits, dim=-1)
    
        labels.extend(label.cpu().numpy())
        preds.extend(predictions.cpu().numpy())
        pred_probs.extend(probabilities[:, 1].cpu().numpy())
    
    return calc_matrics(labels,preds,pred_probs)

def calc_matrics(labels, preds, pred_probs):
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    roc_auc = roc_auc_score(labels, pred_probs)
    f2 = fbeta_score(labels, preds, beta=2)
    return {
        "Accuracy":accuracy,
        "Precision": precision,
        "Recall":recall,
        "F1":f1,
        "ROC-AUC":roc_auc,
        "F2":f2
    }

def reset_model(current_model=None):
    # Delete the old model if it exists to free memory
    if current_model is not None:
        del current_model
        torch.cuda.empty_cache()  # This will free up GPU memory
    
    # Load the new model
    model_name = 'onlplab/alephbert-base'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    
    # Move model to GPU (if available)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    bert_model = bert_model.to(device)
    
    return bert_model

In [12]:
mess_map = {}
conv_map = {}
start = -6
end = 20
for k in range(start,end,2):
    bert_model.train()
    optimizer = torch.optim.AdamW(bert_model.parameters(), lr=learning_rate)
    plan = sigmoid_emphasis_pairs(n=6,k=k)
    for w in plan:
        train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , w, tokenize)
        general_trainer(train_loader, optimizer)
    mess_map[k] = eval(labled_messages_val_loader)
    conv_map[k] = eval(all_messages_val_loader)
    print(("messages",k, mess_map[k]))
    print(("conv",k , conv_map[k]))
    bert_model = reset_model(bert_model)

f1_map_mess = {k:mess_map[k]["F1"] for k in range(start,end,2)}
f1_map_conv = {k:conv_map[k]["F1"] for k in range(start,end,2)}
max_f1_mess = max(f1_map_mess, key=f1_map_mess.get)
max_f1_conv = max(f1_map_conv, key=f1_map_conv.get)

print("max_f1_mess:", max_f1_mess)
print("max_f1_conv:", max_f1_conv)
print("mess_map:", mess_map)
print("conv_map:", conv_map)


bert_model.train()
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=learning_rate)
train_labled_messages_df = pd.concat([train_labled_messages_df, val_labled_messages_df], ignore_index=True)
train_all_messages_df = pd.concat([train_all_messages_df, val_all_messages_df], ignore_index=True)
plan = sigmoid_emphasis_pairs(n=6,k=max_f1_conv)
for w in plan:
    train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , w, tokenize)
    general_trainer(train_loader, optimizer)


Map: 100%|██████████| 1324/1324 [00:00<00:00, 2826.61 examples/s]


('messages', 0, {'Accuracy': 0.924924924924925, 'Precision': 0.5957446808510638, 'Recall': 0.4745762711864407, 'F1': 0.5283018867924528, 'ROC-AUC': 0.92354731522073, 'F2': 0.49469964664310956})
('conv', 0, {'Accuracy': 0.7868852459016393, 'Precision': 0.375, 'Recall': 0.2727272727272727, 'F1': 0.3157894736842105, 'ROC-AUC': 0.7927272727272727, 'F2': 0.28846153846153844})


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 795/795 [00:00<00:00, 2460.50 examples/s]


('messages', 2, {'Accuracy': 0.9144144144144144, 'Precision': 0.5119047619047619, 'Recall': 0.7288135593220338, 'F1': 0.6013986013986014, 'ROC-AUC': 0.9427023706475302, 'F2': 0.671875})
('conv', 2, {'Accuracy': 0.7540983606557377, 'Precision': 0.35714285714285715, 'Recall': 0.45454545454545453, 'F1': 0.4, 'ROC-AUC': 0.7909090909090909, 'F2': 0.43103448275862066})


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 453/453 [00:00<00:00, 2044.22 examples/s]


('messages', 4, {'Accuracy': 0.933933933933934, 'Precision': 0.6666666666666666, 'Recall': 0.5084745762711864, 'F1': 0.5769230769230769, 'ROC-AUC': 0.9340183732164298, 'F2': 0.5338078291814946})
('conv', 4, {'Accuracy': 0.7868852459016393, 'Precision': 0.375, 'Recall': 0.2727272727272727, 'F1': 0.3157894736842105, 'ROC-AUC': 0.78, 'F2': 0.28846153846153844})


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


max_f1_mess: 2
max_f1_conv: 2
mess_map: {0: {'Accuracy': 0.924924924924925, 'Precision': 0.5957446808510638, 'Recall': 0.4745762711864407, 'F1': 0.5283018867924528, 'ROC-AUC': 0.92354731522073, 'F2': 0.49469964664310956}, 2: {'Accuracy': 0.9144144144144144, 'Precision': 0.5119047619047619, 'Recall': 0.7288135593220338, 'F1': 0.6013986013986014, 'ROC-AUC': 0.9427023706475302, 'F2': 0.671875}, 4: {'Accuracy': 0.933933933933934, 'Precision': 0.6666666666666666, 'Recall': 0.5084745762711864, 'F1': 0.5769230769230769, 'ROC-AUC': 0.9340183732164298, 'F2': 0.5338078291814946}}
conv_map: {0: {'Accuracy': 0.7868852459016393, 'Precision': 0.375, 'Recall': 0.2727272727272727, 'F1': 0.3157894736842105, 'ROC-AUC': 0.7927272727272727, 'F2': 0.28846153846153844}, 2: {'Accuracy': 0.7540983606557377, 'Precision': 0.35714285714285715, 'Recall': 0.45454545454545453, 'F1': 0.4, 'ROC-AUC': 0.7909090909090909, 'F2': 0.43103448275862066}, 4: {'Accuracy': 0.7868852459016393, 'Precision': 0.375, 'Recall': 0.27

Map: 100%|██████████| 1019/1019 [00:00<00:00, 2451.27 examples/s]


## eval model

In [13]:
bert_model.eval()
labels = []
preds = []
pred_probs = []

for batch in labled_messages_test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    labels.extend(label.cpu().numpy())
    preds.extend(predictions.cpu().numpy())
    pred_probs.extend(probabilities[:, 1].cpu().numpy())

print(calc_matrics(labels,preds,pred_probs))

{'Accuracy': 0.934612031386225, 'Precision': 0.7397260273972602, 'Recall': 0.4909090909090909, 'F1': 0.5901639344262295, 'ROC-AUC': 0.9159200490926626, 'F2': 0.5263157894736842}


***eval conv***

In [14]:
bert_model.eval()
labels = []
preds = []
pred_probs = []

for batch in all_messages_test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    labels.extend(label.cpu().numpy())
    preds.extend(predictions.cpu().numpy())
    pred_probs.extend(probabilities[:, 1].cpu().numpy())

print(calc_matrics(labels,preds,pred_probs))

{'Accuracy': 0.8524590163934426, 'Precision': 0.5, 'Recall': 0.4444444444444444, 'F1': 0.47058823529411764, 'ROC-AUC': 0.811965811965812, 'F2': 0.45454545454545453}


In [ ]:
torch.save(bert_model.state_dict(), f"model_weights_PCL_mess_conv_opt_plan.pth")